# Markierungs-Export für radinfra.de

Schlankere Fassung von `x_mapillary-markings_generateOutput_2radinfra.ipynb`, im Aufbau
identisch zum Zwilling der Verkehrszeichen-Kampagne
([`../cycleway_complete_campaign/xb_mapillary-trafficsigns_generateOutput_2radinfra.ipynb`](../cycleway_complete_campaign/xb_mapillary-trafficsigns_generateOutput_2radinfra.ipynb)).
Die Logik liegt in [`../cw_campaign.py`](../cw_campaign.py).

> **`x_` läuft wöchentlich auf dem Server** (`scripts/run_mapillary_notebooks.sh`,
> Pipeline `mk`, donnerstags 01:00 UTC) und bleibt unverändert. `xb_` ist die
> Gegenprobe: gleiche Eingabe, gleiche Ausgabe. Erst wenn der Vergleich in der letzten
> Zelle sauber ist, lohnt der Umstieg — und der ist dann eine Zeile im Runner.

**Erzeugt drei Dateien in `mk_output/`**

| Datei | |
| --- | --- |
| `mapillary_markings_bicycle_latest.geojson.gz` | der eigentliche Export |
| `markings_by_month.svg` | Statistikgrafik für die README |
| `README.md` | Beschreibung des Ordners |

**Was gegenüber `x_` anders ist**

1. Laden, Grenzverschnitt, Zeitfilter und Export kommen aus `cw_campaign` — dieselben
   Funktionen, die auch die Verkehrszeichen-Kampagne nutzt.
2. `import mapillary as mly` ist raus — `x_` importiert die Library, benutzt sie nie.
3. Die Daten werden vor dem Lauf von data.vizsim.de geholt, statt auf dem zu rechnen,
   was zufällig lokal liegt.

Die Ausgabe soll **byteweise dieselbe** sein wie bei `x_`. Genau das prüft die letzte Zelle.

## Einstellungen

In [ ]:
# Kein %autoreload hier - anders als 1b_ läuft dieses Notebook unbeaufsichtigt
# per nbconvert auf dem Server.
from pathlib import Path

import pandas as pd

# cw_campaign.py liegt eine Ebene höher, weil beide Cycleway-Kampagnen es nutzen.
# Das ".." funktioniert, weil Jupyter und nbconvert das Arbeitsverzeichnis auf das
# Notebook-Verzeichnis setzen — dieselbe Annahme wie bei ../../output/ und ../utils/.
import sys

sys.path.insert(0, "..")
import cw_campaign as cw

ordner_punkte = Path("../../output")
pfad_metadaten = ordner_punkte / "ml-mf_metadata.json"
pfad_grenze = Path("../utils/OSMB-germany.geojson.gz")
ordner_ausgabe = Path("mk_output")

# Für radinfra.de wird weit zurückgeschaut: hier zählt jede Erkennung, die lange
# genug sichtbar war, nicht nur die aktuellen.
zuletzt_gesehen_nach = "2023-01-01"
# Schutz gegen unplausible Datumsangaben in den Rohdaten.
erstmals_gesehen_nach = "2000-01-01"
# Mindestabstand zwischen erster und letzter Sichtung, in Tagen.
mindest_tage = 180

# Nur die Spalten lesen, die gebraucht werden - die map-feature-points-Parquets
# enthalten alle Feature-Klassen und sind entsprechend groß.
spalten = ["id", "value", "first_seen_at", "last_seen_at", "geometry"]

ordner_ausgabe.mkdir(parents=True, exist_ok=True)
print("Markierungen:", ", ".join(cw.MARKIERUNGEN))

## 1 · Markierungen einlesen

Gefiltert wird **beim Lesen**, je Datei. Das spart RAM — und es bestimmt den
Zeilenindex, der beim Export zur Top-Level-`id` der GeoJSON-Features wird. Wer erst
nach dem Zusammenfügen filtert, bekommt dieselben Zeilen mit anderen ids, und
tippecanoe trägt die in die Vector Tiles.

In [ ]:
metadata = cw.sync_features(ordner_punkte, prefix=cw.PREFIX_MARKIERUNGEN)
print("Datenstand:", metadata["processed_date"])

In [ ]:
# Sollwert aus dem Tile-Cache; ohne Tile-Cache (nur gespiegelte Parquets) None.
erwartet = cw.count_expected_states("../../prep/tile_cache/DE-*_tiles.json")
print("Tile-Cache kennt", erwartet, "Bundesländer")

marks = cw.load_features(
    ordner_punkte,
    prefix=cw.PREFIX_MARKIERUNGEN,
    values=cw.MARKIERUNGEN,
    columns=spalten,
    expect_files=erwartet,
    seen_after=zuletzt_gesehen_nach,
    first_seen_after=erstmals_gesehen_nach,
)
marks.head()

In [ ]:
ml_data_from, processed_date, bundeslaender = cw.read_dataset_metadata(pfad_metadaten)
print("ml_data_from: ", ml_data_from)
print("processed_date:", processed_date)
print("Bundesländer:  ", len(bundeslaender))

## 2 · Filter

In [ ]:
# Die Parquets sind nach Zoom-14-Kacheln geschnitten, Randkacheln ragen ins Ausland.
marks = cw.clip_to_boundary(marks, cw.load_boundary(pfad_grenze))

In [ ]:
# Mindestens zwei Sichtungen mit Abstand - eine einmalige Erkennung kann eine
# Fehlerkennung oder eine temporäre Markierung sein.
marks = cw.filter_by_days_seen(marks, mindest_tage)

## 3 · Export schreiben

In [ ]:
export = cw.add_marking_label(marks)[cw.EXPORT_SPALTEN_MARKIERUNGEN]
pfad_geojson = ordner_ausgabe / "mapillary_markings_bicycle_latest.geojson.gz"

cw.write_geojson_gz(export, pfad_geojson)
cw.assert_ids_are_strings(pfad_geojson)

## 4 · Statistikgrafik

Reine Darstellung, deshalb bewusst im Notebook geblieben und nicht im Modul.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.ticker import FuncFormatter

sns.set_theme(style="whitegrid", context="notebook", rc={
    "axes.facecolor": "#e4e4ed",
    "figure.facecolor": "#DADADF",
    "grid.linestyle": ":",
    "grid.alpha": 0.7,
})

monat = pd.to_datetime(export["last_seen_at"]).dt.to_period("M").dt.to_timestamp()
pro_monat = export.groupby(monat).size()

fig, ax = plt.subplots(figsize=(14, 6))
pro_monat.plot(kind="bar", ax=ax, color="#1f77b4", width=0.7)

ax.set_xticklabels([d.strftime("%Y-%m") for d in pro_monat.index], rotation=45, ha="right")
ax.tick_params(axis="x", labelsize=9)
ax.ticklabel_format(style="plain", axis="y")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:,.0f}".replace(",", ".")))
ax.yaxis.set_major_locator(mtick.MaxNLocator(integer=True))
ax.tick_params(axis="y", labelsize=10)
ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.6)
ax.set_xlabel("Jahr_Monat")
ax.set_ylabel("Anzahl Markierungen")
ax.set_title("Anzahl detected Bicycle Markings nach Monat")

# Summen über den Balken
for i, (monat_i, summe) in enumerate(pro_monat.items()):
    if summe > 0:
        ax.text(i, summe + max(pro_monat) * 0.01, str(int(summe)), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(ordner_ausgabe / "markings_by_month.svg", bbox_inches="tight")
plt.show()

## 5 · README schreiben

In [ ]:
# Der Zeitraum wird bewusst als Zeitstempel formatiert ("2014-03-30 00:00:00"),
# weil x_ die Spalten zu diesem Zeitpunkt als datetime führt. Sieht unschön aus,
# bleibt aber so, damit die Gegenprobe unten greift - ändern lohnt erst nach dem
# Umstieg.
zeitraum = (
    f"{pd.to_datetime(export['first_seen_at']).min()} - {pd.to_datetime(export['last_seen_at']).max()}"
)

readme = cw.build_readme_markierungen(
    anzahl=len(export),
    stand=cw.dataset_stand([processed_date]),
    datensatz_von=cw.dataset_stand([ml_data_from], fallback="N/A"),
    zeitraum=zeitraum,
    seit=zuletzt_gesehen_nach,
    min_days=mindest_tage,
)
(ordner_ausgabe / "README.md").write_text(readme, encoding="utf-8")
print(readme[:420])

## 6 · Gegenprobe gegen `x_`

Setzt einen Referenzlauf von `x_` in `mk_output_ref/` voraus:

```bash
jupyter nbconvert --to notebook --execute x_mapillary-markings_generateOutput_2radinfra.ipynb
cp -r mk_output mk_output_ref
```

Verglichen wird die entpackte GeoJSON — der gzip-Rohbytevergleich taugt nicht, weil
im gzip-Header ein Zeitstempel steckt.

In [ ]:
import gzip
import hashlib
import json
import re

ordner_referenz = Path("mk_output_ref")


def geojson_fingerabdruck(pfad):
    with gzip.open(pfad, "rt", encoding="utf-8") as f:
        roh = f.read()
    daten = json.loads(roh)
    return {
        "sha256": hashlib.sha256(roh.encode("utf-8")).hexdigest(),
        "features": len(daten["features"]),
        "spalten": list(daten["features"][0]["properties"].keys()),
        "erste_id": daten["features"][0].get("id"),
        "letzte_id": daten["features"][-1].get("id"),
    }


def svg_ohne_rauschen(pfad):
    # matplotlib schreibt einen Zeitstempel und würfelt die clip-path-ids pro
    # Prozess neu; zwei Läufe von x_ unterscheiden sich darin genauso.
    s = pfad.read_text(encoding="utf-8")
    s = re.sub(r"<dc:date>[^<]*</dc:date>", "<dc:date/>", s)
    return re.sub(r"\b[a-z]p?[0-9a-f]{8,}\b", "ID", s)


name_gz = "mapillary_markings_bicycle_latest.geojson.gz"
if not ordner_referenz.exists():
    print(f"{ordner_referenz} fehlt - x_ laufen lassen und mk_output dorthin kopieren")
else:
    a = geojson_fingerabdruck(ordner_referenz / name_gz)
    b = geojson_fingerabdruck(ordner_ausgabe / name_gz)
    for schluessel in a:
        gleich = "OK  " if a[schluessel] == b[schluessel] else "NEIN"
        print(f"  {gleich} {schluessel}")
        if a[schluessel] != b[schluessel]:
            print(f"       x_:  {a[schluessel]}")
            print(f"       xb_: {b[schluessel]}")

    alt = (ordner_referenz / "README.md").read_text(encoding="utf-8")
    neu = (ordner_ausgabe / "README.md").read_text(encoding="utf-8")
    print(f"  {'OK  ' if alt == neu else 'NEIN'} README.md")

    gleich = svg_ohne_rauschen(ordner_referenz / "markings_by_month.svg") == svg_ohne_rauschen(
        ordner_ausgabe / "markings_by_month.svg"
    )
    print(f"  {'OK  ' if gleich else 'NEIN'} markings_by_month.svg (ohne Zeitstempel und clip-path-ids)")